In [1]:
# Install required packages if needed
!pip install pymannkendall statsmodels -q

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from statsmodels.tsa.seasonal import STL
import pymannkendall as mk

import warnings
warnings.filterwarnings("ignore")

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
df = pd.read_csv("/content/drive/MyDrive/sih/groundwater_clean_final.csv")

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

df.head()

Dataset shape: (36117, 4)

Columns:
['Station', 'Timestamp', 'Groundwater_Level_Final', 'Quality_Flag']


,Station,Timestamp,Groundwater_Level_Final,Quality_Flag
0,Bajengdoba_1,2023-05-04 18:00:00,-5.51,OK
1,Bajengdoba_1,2023-05-05 00:00:00,-5.52,OK
2,Bajengdoba_1,2023-05-05 06:00:00,-5.49,OK
3,Bajengdoba_1,2023-05-05 12:00:00,-5.51,OK
4,Bajengdoba_1,2023-05-05 18:00:00,-5.46,OK


In [6]:
df["Timestamp"] = pd.to_datetime(df["Timestamp"])

df["Groundwater_Level_Final"] = pd.to_numeric(
    df["Groundwater_Level_Final"],
    errors="coerce"
)

# Sort chronologically within each station
df = df.sort_values(["Station", "Timestamp"]).reset_index(drop=True)

print("Date range:")
print(df["Timestamp"].min(), "to", df["Timestamp"].max())

print("\nNumber of stations:", df["Station"].nunique())

print("\nStations:")
print(df["Station"].unique())

Date range:
2023-05-01 00:00:00 to 2025-12-31 18:00:00

Number of stations: 15

Stations:
['Bajengdoba_1' 'Chasingre (NEHU)' 'Dadenggre_1' 'Jengjal' 'Mendal_1'
 'Mendipathar_1' 'NIT Cherrapunji' 'Nongladew_1' 'Nongladew_2'
 'Powergrid Khliehriat' 'Purduwa_1' 'RPBF Kyrdemkulai_1' 'Rongram_1'
 'Tdohumshiaw' 'W.R.D., Shillong']


In [7]:
quality_summary = df["Quality_Flag"].value_counts(dropna=False)

print("QUALITY FLAG SUMMARY")
print("=" * 50)
print(quality_summary)

print("\nMissing final values:")
print(df["Groundwater_Level_Final"].isna().sum())

print("\nDuplicate Station-Timestamp records:")
print(df.duplicated(subset=["Station", "Timestamp"]).sum())

QUALITY FLAG SUMMARY
Quality_Flag
OK                30347
GAP                5766
SHORT_INTERVAL        4
Name: count, dtype: int64

Missing final values:
5127

Duplicate Station-Timestamp records:
0


In [8]:
trend_df = df.dropna(
    subset=["Groundwater_Level_Final"]
).copy()

print("Records available for trend analysis:", len(trend_df))

print("\nRecords by station:")
print(
    trend_df.groupby("Station")
    .size()
    .sort_values(ascending=False)
)

Records available for trend analysis: 30990

Records by station:
Station
W.R.D., Shillong        3264
Purduwa_1               3236
NIT Cherrapunji         3234
Nongladew_1             3220
Powergrid Khliehriat    2580
Bajengdoba_1            2486
Mendipathar_1           2266
Mendal_1                2139
Rongram_1               2100
RPBF Kyrdemkulai_1      1603
Dadenggre_1             1598
Nongladew_2             1308
Jengjal                  880
Tdohumshiaw              868
Chasingre (NEHU)         208
dtype: int64
